## A1. Imports + Baseline Logistic Regression

In [4]:
import numpy as np
import pandas as pd

df = pd.read_csv('../data/adult/adult_with_pii.csv')

In [28]:
from sklearn.model_selection import train_test_split

# df['Target'] = df['Target'].apply(lambda x: 0 if x[0] == '<' else 1)
df = df.rename(columns={"Martial Status": "Marital Status"})
df


,Name,DOB,SSN,Zip,Age,Workclass,fnlwgt,Education,Education-Num,Marital Status,Occupation,Relationship,Race,Sex,Capital Gain,Capital Loss,Hours per week,Country,Target
0,Karrie Trusslove,9/7/67,732-14-6110,64152,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,Brandise Tripony,6/7/88,150-19-2766,61523,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,Brenn McNeely,8/6/91,725-59-9860,95668,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,Dorry Poter,4/6/09,659-57-4974,25503,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,Dick Honnan,9/16/51,220-93-3811,75387,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,Ardyce Golby,10/29/61,212-61-8338,41328,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,0
32557,Jean O'Connor,6/28/52,737-32-2919,94735,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,1
32558,Reuben Skrzynski,8/9/66,314-48-0219,49628,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,0
32559,Caye Biddle,5/19/78,647-75-3550,8213,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,0


In [ ]:
X = df.drop(columns=['Name', 'SSN', 'Zip', 'Target', 'fnlwgt', 'DOB', 'Education', 'Relationship'])
y = df[['Target']]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1, test_size=0.2)
X_train

In [18]:
X_test

,Age,Workclass,Education-Num,Marital Status,Occupation,Race,Sex,Capital Gain,Capital Loss,Hours per week,Country
9646,62,Self-emp-not-inc,4,Widowed,Other-service,White,Female,0,0,66,United-States
709,18,Private,7,Never-married,Other-service,White,Male,0,0,25,United-States
7385,25,Private,13,Never-married,Farming-fishing,White,Male,27828,0,50,United-States
16671,33,Private,9,Married-civ-spouse,Prof-specialty,White,Male,0,0,40,United-States
21932,36,Private,7,Never-married,Machine-op-inspct,White,Female,0,0,40,United-States
...,...,...,...,...,...,...,...,...,...,...,...
5889,39,Private,13,Married-civ-spouse,Prof-specialty,White,Female,0,0,20,United-States
25723,17,Private,6,Never-married,Sales,White,Female,0,0,20,United-States
29514,35,Private,9,Never-married,Transport-moving,White,Male,0,0,40,United-States
1600,30,Private,7,Married-civ-spouse,Craft-repair,White,Male,0,0,45,United-States


In [19]:
# get different var types
integer_vars = ['Education-Num', 'Age', 'Capital Gain', 'Capital Loss', 'Hours per week']
categorical_vars = ['Workclass', 'Marital Status', 'Occupation', 'Race', 'Country']
binary_vars = ['Sex'] # don't include Target

# do vanilla Logistic Regression

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Create the preprocessing stages
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), integer_vars),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_vars),
        ("bin", OneHotEncoder(drop="if_binary"), binary_vars),
    ],
    remainder="passthrough",  # This ensures target 'income' is kept unchanged
)

# 2. Construct the  pipeline
vanilla_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(solver="lbfgs")),
    ]
)

vanilla_pipeline.fit(X_train, y_train)

# 3. Make predictions
predictions = vanilla_pipeline.predict(X_test)

/Users/ammareltigani/miniconda3/envs/erdos_project_environment/lib/python3.12/site-packages/sklearn/utils/validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [20]:
from sklearn.metrics import accuracy_score, precision_score

# 4. Get accuracy and precision

accuracy = accuracy_score(y_test, predictions)
# Precision: Out of all positive predictions, how many were actually positive?
precision = precision_score(y_test, predictions)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")

Accuracy:  0.8529
Precision: 0.7033


In [55]:
import pandas as pd
import numpy as np

# extract coefficients and matching feature names from the trained pipeline

clf = vanilla_pipeline.named_steps["classifier"]
coefs = clf.coef_

# handle binary vs multiclass shapes
if coefs.ndim == 2 and coefs.shape[0] == 1:
    coefs = coefs.ravel()

# try to get feature names from the preprocessor (sklearn >=1.0)
try:
    feature_names = preprocessor.get_feature_names_out(X_train.columns)
    feature_names = list(feature_names)
except Exception:
    # fallback: construct names manually from known variable lists and encoders
    feature_names = []
    # numeric features (StandardScaler)
    feature_names.extend(integer_vars)
    # categorical features (OneHotEncoder)
    cat_enc = preprocessor.named_transformers_.get("cat")
    if cat_enc is not None and hasattr(cat_enc, "get_feature_names_out"):
        feature_names.extend(cat_enc.get_feature_names_out(categorical_vars).tolist())
    else:
        # if encoder not available, include raw names as fallback
        feature_names.extend(categorical_vars)
    # binary features (OneHotEncoder with drop)
    bin_enc = preprocessor.named_transformers_.get("bin")
    if bin_enc is not None and hasattr(bin_enc, "get_feature_names_out"):
        feature_names.extend(bin_enc.get_feature_names_out(binary_vars).tolist())
    else:
        feature_names.extend(binary_vars)
    # add any passthrough columns that were not listed above
    remainder_cols = [c for c in X_train.columns if c not in feature_names and c not in integer_vars + categorical_vars + binary_vars]
    feature_names.extend(remainder_cols)

# ensure lengths match; if not, try best-effort alignment
if len(feature_names) != len(coefs):
    # attempt to trim or pad feature_names to match coef length
    if len(feature_names) > len(coefs):
        feature_names = feature_names[: len(coefs)]
    else:
        feature_names = feature_names + [f"extra_{i}" for i in range(len(coefs) - len(feature_names))]

# build dataframe of coefficients
if isinstance(coefs, np.ndarray) and coefs.ndim == 1:
    coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefs})
else:
    # multiclass: create columns per class
    cols = {f"class_{i}": coefs[i] for i in range(coefs.shape[0])}
    coef_df = pd.DataFrame({ "feature": feature_names, **{k: v for k, v in cols.items()} })

# show coefficients (unsorted) and also top features by absolute magnitude
print("All feature coefficients:")
display(coef_df)

print("\nTop features by absolute coefficient value:")
if "coefficient" in coef_df.columns:
    display(coef_df.reindex(coef_df.coefficient.abs().sort_values(ascending=True).index).head(20))
else:
    # multiclass: show sum of absolute coeffs as importance
    abs_sum = coef_df.drop(columns="feature").abs().sum(axis=1)
    display(pd.concat([coef_df, pd.Series(abs_sum, name="abs_sum")], axis=1).sort_values("abs_sum", ascending=True).head(20))

All feature coefficients:


,feature,coefficient
0,num__Education-Num,0.722722
1,num__Age,0.341859
2,num__Capital Gain,2.311849
3,num__Capital Loss,0.261735
4,num__Hours per week,0.375696
...,...,...
78,cat__Country_United-States,0.357203
79,cat__Country_Vietnam,-0.722253
80,cat__Country_Yugoslavia,-0.014551
81,cat__Country_nan,0.050559



Top features by absolute coefficient value:


,feature,coefficient
57,cat__Country_Hungary,-0.001379
8,cat__Workclass_Private,0.010295
80,cat__Country_Yugoslavia,-0.014551
37,cat__Race_Asian-Pac-Islander,-0.016002
47,cat__Country_Ecuador,0.018911
77,cat__Country_Trinadad&Tobago,0.021272
76,cat__Country_Thailand,-0.024582
75,cat__Country_Taiwan,0.036831
81,cat__Country_nan,0.050559
70,cat__Country_Poland,0.059334


## D1. $k$-anonymity on original data

In [ ]:
dfc = df.drop(columns=['Name', 'SSN', 'Zip', 'DOB', 'Target', 'fnlwgt'])

d = {}
for i in range(len(dfc)):
    row = tuple(dfc.iloc[i])
    if row in d:
        d[row].append(i)
    else:
        d[row] = [i]

anonymity_classes = np.array([len(v) for _,v in d.items()])

# indices_of_max_rows = [v for _,v in d.items() if len(v) == 20]
# print(indices_of_max_rows)
# print(dfc.iloc[2457], dfc.iloc[5078])

print(anonymity_classes.max())
print(anonymity_classes.min())
print(anonymity_classes.mean())

    

20
1
1.1428120174083953


## D1.5 $l$-diversity, $t$-closeness

## A2. Adding uniform noise, resampling, and binning